In [2]:
# ==============================================
# START OF SCRIPT
# ==============================================

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # Suppresses TensorFlow CPU warnings

import faiss
import numpy as np
import logging
import ipywidgets as widgets
import boto3
import json
import time
from concurrent.futures import ThreadPoolExecutor
from threading import Lock
from typing import List
from IPython.display import display
from functools import lru_cache
from sentence_transformers import SentenceTransformer

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Configuration Dictionary (Only Selected Models)
CONFIG = {
    "bucket_name": "sagemaker-us-east-1-188494237500",
    "json_folder": "dev/json",
    "index_folder": "dev/idx",
    "nlist": 512,
    "nprobe": 16,
    "num_training_samples": 1000,
    "embedding_models": {
        "amazon.titan-embed-text-v2:0": 1536,
        "anthropic.claude-3-sonnet-20240229-v1:0": 8192,
        "meta.llama3-70b-instruct-v1:0": 8192
    },
    "default_model": "amazon.titan-embed-text-v2:0"
}

# Initialize S3 and Bedrock clients

# This function requires ObjectTagging Read/Write permissions
def check_s3_object_tag(bucket: str, key: str, tag_key: str) -> bool:
    """Checks if an object in S3 has a specific tag."""
    try:
        response = s3.get_object_tagging(Bucket=bucket, Key=key)
        tags = {tag["Key"]: tag["Value"] for tag in response.get("TagSet", [])}
        return tag_key in tags
    except s3.exceptions.ClientError as e:
        if e.response['Error']['Code'] == 'AccessDenied':
            logging.warning(f"Skipping tag check for {key} due to insufficient permissions.")
            return False  # Assume the object has no tag and continue
        else:
            logging.error(f"Error retrieving tags for {key}: {e}")
            return False

def list_objects_s3(bucket: str, prefix: str) -> List[str]:
    """Lists all objects in an S3 bucket under a given prefix."""
    try:
        response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
        return [obj["Key"] for obj in response.get("Contents", [])]
    except Exception as e:
        logging.error(f"⚠️ Error listing S3 objects: {e}")
        return []
s3_client = boto3.client("s3")

# Ensure S3 client is initialized before being used
s3 = boto3.client("s3")
bedrock = boto3.client("bedrock-runtime")
query_cache = {}
faiss_lock = Lock()  # Lock for FAISS index updates

# Load FAISS index from S3
def load_faiss_index(model_name):
    bucket = CONFIG["bucket_name"]
    index_key = f"{CONFIG['index_folder']}/{model_name}.faiss"
    
    try:
        response = s3_client.get_object(Bucket=bucket, Key=index_key)
        index_data = np.frombuffer(response["Body"].read(), dtype=np.uint8)
        index = faiss.deserialize_index(index_data)
        logger.info(f"✅ Loaded FAISS index for {model_name} from S3.")
    except s3_client.exceptions.NoSuchKey:
        logger.info(f"⚠️ No existing FAISS index found for {model_name}, creating a new one. This may take some time...")
        index = faiss.IndexFlatL2(CONFIG["embedding_models"][model_name])
    except Exception as e:
        logger.error(f"❌ Error loading FAISS index for {model_name}: {e}. Creating a new one.")
        index = faiss.IndexFlatL2(CONFIG["embedding_models"][model_name])
    
    return index, index_key

# Save FAISS index to S3
def save_faiss_index(index, index_key):
    with faiss_lock:  # Prevent multiple threads from writing at the same time
        try:
            index_data = faiss.serialize_index(index).tobytes()
            s3_client.put_object(Bucket=CONFIG["bucket_name"], Key=index_key, Body=index_data)
            logger.info(f"✅ FAISS index saved to {index_key}")
        except Exception as e:
            logger.error(f"❌ Failed to save FAISS index to {index_key}: {e}")

# Bedrock Embedding Function with Cache
def generate_embedding(text, model_name):
    cache_key = (text, model_name)
    
    if cache_key in query_cache:
        logger.info(f"⚡ Using cached embedding for '{text}' ({model_name})")
        return query_cache[cache_key]
    
    try:
        logger.info("⚡ Generating embedding via AWS Bedrock...")
        start_time = time.time()
        response = bedrock.invoke_model(
            modelId=model_name,
            contentType="application/json",
            accept="application/json",
            body=json.dumps({"inputText": text})
        )
        elapsed = time.time() - start_time
        logger.info(f"🕒 Bedrock embedding took {elapsed:.2f} seconds.")
        if 'Body' in response and response['Body']:
            response_body = json.loads(response['Body'].read().decode('utf-8'))
        else:
            logger.error(f"❌ Bedrock response for {model_name} did not contain a valid body.")
            return []
        embedding = response_body.get('embedding', [])
    except Exception as e:
        logger.error(f"❌ Error generating embedding for {model_name}: {e}")
        return []

    expected_dim = CONFIG["embedding_models"].get(model_name, None)
    if expected_dim and len(embedding) != expected_dim:
        logger.warning(f"⚠️ Embedding dimension mismatch for {model_name}: Expected {expected_dim}, got {len(embedding)}")
        return []

    query_cache[cache_key] = embedding
    return embedding

# Process a single JSON file
def clean_text(text: str) -> str:
    """Cleans extracted text by removing unwanted characters and normalizing spaces."""
    return text.replace('
', ' ').replace('	', ' ').strip()
    """Cleans extracted text by removing unwanted characters and normalizing spaces."""
    return text.replace('
', ' ').replace('	', ' ').strip()

def process_single_file(obj, model_name):
    bucket = CONFIG["bucket_name"]
    file_key = obj["Key"]
    
    try:
        file_obj = s3_client.get_object(Bucket=bucket, Key=file_key)
        file_content_str = file_obj["Body"].read().decode("utf-8").strip()
        if not file_content_str:
            logger.warning(f"⚠️ Empty JSON file: {file_key}. Skipping.")
            return
        file_content = json.loads(file_content_str)
    except json.JSONDecodeError:
        logger.error(f"❌ Invalid JSON format in {file_key}. Skipping.")
        return
    except Exception as e:
        logger.error(f"❌ Failed to load {file_key}: {e}")
        return
    
    # ✅ Combine all text from chunks to generate a single document-level embedding
    combined_text = " ".join(clean_text(chunk.get("text", "").strip()) for chunk in file_content.get("chunks", []))
    
    if combined_text:
        if "embeddings" not in file_content:
            file_content["embeddings"] = {}
        file_content["embeddings"][model_name] = generate_embedding(combined_text, model_name)
    
    # ✅ Remove per-chunk embeddings to avoid redundancy
    for chunk in file_content.get("chunks", []):
        chunk.pop("embedding", None)

    if "chunks" in file_content:
        for chunk in file_content["chunks"]:
            text_data = clean_text(chunk.get("text", "").strip())
            if text_data:
                if "embedding" not in chunk:
                    chunk["embedding"] = {}
                chunk["embedding"][model_name] = generate_embedding(text_data, model_name)
    
    s3_client.put_object(Bucket=bucket, Key=file_key, Body=json.dumps(file_content))
    logger.info(f"✅ Updated embeddings in {file_key}")

# Process JSON files and update embeddings
def process_json_files(model_name):
    start_time = time.time()
    logger.info(f"🚀 Starting processing for model {model_name}...")
    bucket = CONFIG["bucket_name"]
    folder = CONFIG["json_folder"]
    index, index_key = load_faiss_index(model_name)
    
    logger.info("🔍 Retrieving file list from S3...")
    response = s3_client.list_objects_v2(Bucket=bucket, Prefix=folder)
    if "Contents" not in response or len(response["Contents"]) == 0:
        logger.error("❌ No JSON files found in S3 bucket. Ensure the folder is populated. No processing was performed.")
        return
    
    with ThreadPoolExecutor(max_workers=10) as executor:
        executor.map(process_single_file, response["Contents"], [model_name] * len(response["Contents"]))
    
    save_faiss_index(index, index_key)
    duration = time.time() - start_time
    logger.info(f"Finished processing all JSON files in {duration:.2f} seconds. Total files processed: {len(response['Contents'])}")

# UI Selector
def on_model_change(change):
    selected_model = change['new']
    logger.info(f"Model selected: {selected_model}")
    process_json_files(selected_model)

execute_button = widgets.Button(
    description="Execute",
    button_style="success"
)
execute_button.on_click(lambda _: process_json_files(model_selector.value))

model_selector = widgets.Dropdown(
    options=list(CONFIG["embedding_models"].keys()),
    value=CONFIG["default_model"],
    description="Embedding Model:",
)
model_selector.observe(on_model_change, names='value')

# Display the UI
display(model_selector, execute_button)

# ==============================================
# END OF SCRIPT
# ==============================================


SyntaxError: unterminated string literal (detected at line 144) (3349443399.py, line 144)